In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

##Clean Data

In [2]:
df = pd.read_csv("global_bleaching_environmental.csv")

/tmp/ipykernel_8119/3241338298.py:1: DtypeWarning: Columns (13,15,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("global_bleaching_environmental.csv")


In [5]:
df.shape

(41361, 62)

In [ ]:
numeric_cols = ['Latitude_Degrees', 'Longitude_Degrees', 'Distance_to_Shore',
       'Turbidity', 'Cyclone_Frequency', 'Date_Year', 'Depth_m',
       'Percent_Cover', 'Bleaching_Level', 'Percent_Bleaching', 'ClimSST',
       'Temperature_Kelvin', 'Temperature_Mean', 'Temperature_Minimum',
       'Temperature_Maximum', 'Temperature_Kelvin_Standard_Deviation',
       'Windspeed', 'SSTA', 'SSTA_Standard_Deviation', 'SSTA_Mean',
       'SSTA_Minimum', 'SSTA_Maximum', 'SSTA_Frequency',
       'SSTA_Frequency_Standard_Deviation', 'SSTA_FrequencyMax',
       'SSTA_FrequencyMean', 'SSTA_DHW', 'SSTA_DHW_Standard_Deviation',
       'SSTA_DHWMax', 'SSTA_DHWMean', 'TSA', 'TSA_Standard_Deviation',
       'TSA_Minimum', 'TSA_Maximum', 'TSA_Mean', 'TSA_Frequency',
       'TSA_Frequency_Standard_Deviation', 'TSA_FrequencyMax',
       'TSA_FrequencyMean', 'TSA_DHW', 'TSA_DHW_Standard_Deviation',
       'TSA_DHWMax', 'TSA_DHWMean',]

drop_columns = ['Site_ID', 'Sample_ID', 'Data_Source','Ocean_Name', 'Reef_ID', 'Realm_Name',
       'Ecoregion_Name', 'Country_Name', 'State_Island_Province_Name',
       'City_Town_Name', 'Site_Name', 'Date_Day',
       'Substrate_Name', 'Site_Comments', 'Sample_Comments',
       'Bleaching_Comments','Exposure', 'Bleaching_Level', 'Percent_Cover',
       'Distance_to_Shore','Turbidity','Cyclone_Frequency','SSTA_Mean'
       ]

df.drop(columns= drop_columns, inplace= True)

for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df[df['Date_Year'] >= 1997]
print(df['Date_Year'].value_counts().sort_index())

In [ ]:
# Filter
df = df[df['SSTA'] > 0]
df = df.dropna()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
rename_map = {
    'Date_Month': 'Month',
    'Date_Year': 'Year',
    'Depth_m': 'Depth',
    'Percent_Bleaching': 'Average_Bleaching'
}

df = df.rename(columns=rename_map)

In [ ]:
df = df.sort_values(['Latitude_Degrees', 'Longitude_Degrees', 'Date'])
df['SSTA_Rolling_Mean'] = df.groupby(['Latitude_Degrees', 'Longitude_Degrees'])['SSTA'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['TSA_Rolling_Mean'] = df.groupby(['Latitude_Degrees', 'Longitude_Degrees'])['TSA'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())


###  One-Hot Encoding for Month

In [ ]:
month_dummies = pd.get_dummies(df['Month'], prefix='Month', drop_first=False)
df = pd.concat([df, month_dummies], axis=1)
df = df.drop(columns=['Month', 'Date', 'Year'])

# PCA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
X = df.drop("Average_Bleaching", axis=1)
Y = df["Average_Bleaching"]

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
X_train_raw_month, X_test_raw_month = X_train_raw[['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']].copy(), X_test_raw[['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']].copy()

X_train_raw = X_train_raw.drop(columns= ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12'])
X_test_raw = X_test_raw.drop(columns= ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12'])

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

In [ ]:
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

# --- Plot 1: Cumulative Variance (your existing plot) ---
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_components_95 = np.argmax(cumvar >= 0.95) + 1

ax1.plot(range(1, len(cumvar)+1), cumvar, marker='o', linewidth=2)
ax1.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
ax1.axvline(x=n_components_95, color='g', linestyle='--',
            label=f'n={n_components_95} components')
ax1.set_xlabel('Number of Components')
ax1.set_ylabel('Cumulative Variance')
ax1.set_title('PCA - Cumulative Variance')
ax1.legend()
ax1.grid(True)

# --- Plot 2: Individual variance per component (bar chart) ---
ax2.bar(range(1, len(pca.explained_variance_ratio_)+1),
        pca.explained_variance_ratio_ * 100,
        color='steelblue', alpha=0.7, edgecolor='black')
ax2.axvline(x=n_components_95, color='g', linestyle='--',
            label=f'95% threshold at n={n_components_95}')
ax2.set_xlabel('Principal Component')
ax2.set_ylabel('Variance Explained (%)')
ax2.set_title('PCA - Individual Variance per Component')
ax2.legend()
ax2.grid(True, axis='y')

plt.tight_layout()
plt.savefig('pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Components needed for 95% variance: {n_components_95}")
print(f"✅ Variance explained by {n_components_95} components: {cumvar[n_components_95-1]:.4f}")

In [ ]:
# Convert X_train_pca and X_test_pca to DataFrames first
X_train_pca_df = pd.DataFrame(X_train_pca, columns=[f'PC{i+1}' for i in range(X_train_pca.shape[1])])
X_test_pca_df = pd.DataFrame(X_test_pca, columns=[f'PC{i+1}' for i in range(X_test_pca.shape[1])])

# Reset index on month DataFrames to align with PCA DataFrames
X_train_raw_month = X_train_raw_month.reset_index(drop=True)
X_test_raw_month = X_test_raw_month.reset_index(drop=True)

# Now concat safely
X_train_pca = pd.concat([X_train_pca_df, X_train_raw_month], axis=1)
X_test_pca = pd.concat([X_test_pca_df, X_test_raw_month], axis=1)

# lazyPredict

In [ ]:
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

In [ ]:
reg = LazyRegressor(verbose=1, ignore_warnings=True, custom_metric=None)
models, predictions = reg.fit(X_train_pca, X_test_pca, y_train, y_test)

print(models)

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

# OpTuna

In [26]:
#prepare a version of X_train that already has month columns
month_cols = ['Month_1','Month_2','Month_3','Month_4','Month_5','Month_6',
              'Month_7','Month_8','Month_9','Month_10','Month_11','Month_12']

X_train_no_month = X_train_raw.copy()   # numeric only (month already dropped earlier)
X_train_month    = X_train_raw_month.reset_index(drop=True)
# helper: run CV with month columns reattached after PCA inside each fold
def cv_with_month(pipeline, X_num, X_mon, y, cv=4):
    from sklearn.model_selection import KFold

    # Reset all indices upfront so iloc slicing stays aligned
    X_num = X_num.reset_index(drop=True)
    X_mon = X_mon.reset_index(drop=True)
    y     = y.reset_index(drop=True)

    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X_num):
        X_num_tr, X_num_val = X_num.iloc[train_idx], X_num.iloc[val_idx]
        X_mon_tr  = X_mon.iloc[train_idx].reset_index(drop=True)
        X_mon_val = X_mon.iloc[val_idx].reset_index(drop=True)
        y_tr  = y.iloc[train_idx].reset_index(drop=True)  # ← fixed
        y_val = y.iloc[val_idx].reset_index(drop=True)    # ← fixed

        steps_no_model = Pipeline(pipeline.steps[:-1])
        X_tr_pca  = steps_no_model.fit_transform(X_num_tr)
        X_val_pca = steps_no_model.transform(X_num_val)

        n_pcs   = X_tr_pca.shape[1]
        pc_cols = [f'PC{i+1}' for i in range(n_pcs)]

        X_tr_final  = pd.concat([pd.DataFrame(X_tr_pca,  columns=pc_cols), X_mon_tr],  axis=1)
        X_val_final = pd.concat([pd.DataFrame(X_val_pca, columns=pc_cols), X_mon_val], axis=1)

        model = pipeline.steps[-1][1]
        model.fit(X_tr_final, y_tr)
        scores.append(model.score(X_val_final, y_val))
    return np.mean(scores)

In [ ]:
def et_objective(trial):
    model = ExtraTreesRegressor(
        n_estimators     = trial.suggest_int  ("n_estimators",  50, 400),
        max_depth        = trial.suggest_int  ("max_depth",      5,  40),
        min_samples_leaf = trial.suggest_int  ("min_samples_leaf",1,  5),
        max_features     = trial.suggest_categorical("max_features", ["sqrt","log2"]),
        n_jobs=-1, random_state=42
    )
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('pca',     PCA(n_components=0.95)),
        ('model',   model)
    ])
    return cv_with_month(pipeline, X_train_no_month, X_train_month, y_train)

et_study = optuna.create_study(direction="maximize")
et_study.optimize(et_objective, n_trials=50)
print("========== ET BEST ==========")
print(et_study.best_params)
print(f"Best R²: {et_study.best_value:.4f}")

In [27]:
def xgb_objective(trial):
    model = xgb.XGBRegressor(
        n_estimators      = trial.suggest_int  ("n_estimators",   50, 400),
        learning_rate     = trial.suggest_float("learning_rate", 0.005, 0.1),
        max_depth         = trial.suggest_int  ("max_depth",       3,  15),
        subsample         = trial.suggest_float("subsample",      0.5, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree",0.5, 1.0),
        n_jobs=-1, random_state=42
    )
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('pca',     PCA(n_components=0.95)),
        ('model',   model)
    ])
    return cv_with_month(pipeline, X_train_no_month, X_train_month, y_train)

xgb_study = optuna.create_study(direction="maximize")
xgb_study.optimize(xgb_objective, n_trials=50)
print("========== XGB BEST ==========")
print(xgb_study.best_params)
print(f"Best R²: {xgb_study.best_value:.4f}")

[I 2026-05-05 14:08:00,755] A new study created in memory with name: no-name-e4a70631-e7af-4f7e-b5b1-78d99c27e5fc
[I 2026-05-05 14:08:24,044] Trial 0 finished with value: 0.6269646241745608 and parameters: {'n_estimators': 236, 'learning_rate': 0.0784716143512788, 'max_depth': 10, 'subsample': 0.8749139989256398, 'colsample_bytree': 0.6467368513148106}. Best is trial 0 with value: 0.6269646241745608.
[I 2026-05-05 14:08:58,834] Trial 1 finished with value: 0.6263887815308664 and parameters: {'n_estimators': 197, 'learning_rate': 0.050678028666263135, 'max_depth': 12, 'subsample': 0.6595283746097667, 'colsample_bytree': 0.6855795794668653}. Best is trial 0 with value: 0.6269646241745608.
[I 2026-05-05 14:09:03,523] Trial 2 finished with value: 0.35002043127558424 and parameters: {'n_estimators': 100, 'learning_rate': 0.00977144285775199, 'max_depth': 7, 'subsample': 0.5192786400845714, 'colsample_bytree': 0.9092487601017184}. Best is trial 0 with value: 0.6269646241745608.
[I 2026-05-05

========== XGB BEST ==========
{'n_estimators': 400, 'learning_rate': 0.024294864577658625, 'max_depth': 12, 'subsample': 0.6937835791180113, 'colsample_bytree': 0.7141452636449821}
Best R²: 0.6313


In [28]:
def rf_objective(trial):
    model = RandomForestRegressor(
        n_estimators      = trial.suggest_int("n_estimators", 50, 500),
        max_depth         = trial.suggest_int("max_depth", 5, 30),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 10),
        max_features      = trial.suggest_float("max_features", 0.1, 1.0),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('pca',     PCA(n_components=0.95)),
        ('model',   model)
    ])

    return cv_with_month(pipeline, X_train_no_month, X_train_month, y_train)

rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(rf_objective, n_trials=50)

print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

[I 2026-05-05 14:36:00,588] A new study created in memory with name: no-name-90c5245f-10e0-4fe6-b131-b14221ab77f1
[I 2026-05-05 14:36:05,457] Trial 0 finished with value: 0.6088340767685325 and parameters: {'n_estimators': 216, 'max_depth': 22, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.2717579802643843}. Best is trial 0 with value: 0.6088340767685325.
[I 2026-05-05 14:36:26,327] Trial 1 finished with value: 0.5388062468540961 and parameters: {'n_estimators': 328, 'max_depth': 26, 'min_samples_split': 9, 'min_samples_leaf': 7, 'max_features': 0.9727265174203735}. Best is trial 0 with value: 0.6088340767685325.
[I 2026-05-05 14:36:28,550] Trial 2 finished with value: 0.3633797817534631 and parameters: {'n_estimators': 99, 'max_depth': 6, 'min_samples_split': 9, 'min_samples_leaf': 9, 'max_features': 0.665638854688708}. Best is trial 0 with value: 0.6088340767685325.
[I 2026-05-05 14:36:33,760] Trial 3 finished with value: 0.5541849115817792 and parameters: {'n_esti

========== RF BEST ==========
{'n_estimators': 422, 'max_depth': 23, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.4576285822812347}
Best R²: 0.6156


# Start

In [29]:
hgb_model = HistGradientBoostingRegressor(
    max_iter=211,
    max_depth=10,
    learning_rate=0.0485,
    min_samples_leaf=11,
    l2_regularization=0.8831,
    random_state=42
)
# {'max_iter': 211, 'max_depth': 10, 'learning_rate': 0.04846251191044525, 'min_samples_leaf': 11, 'l2_regularization': 0.8830944711161548}

et_model = ExtraTreesRegressor(
    n_estimators=254,
    max_depth=40,
    min_samples_leaf=1,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
#{'n_estimators': 254, 'max_depth': 40, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

xgb_model = xgb.XGBRegressor(
    n_estimators=400,
    learning_rate=0.0243,
    max_depth=12,
    subsample=0.694,
    colsample_bytree=0.714,
    n_jobs=-1,
    random_state=42
)
#{'n_estimators': 400, 'learning_rate': 0.024294864577658625, 'max_depth': 12, 'subsample': 0.6937835791180113, 'colsample_bytree': 0.7141452636449821}

rf_model = RandomForestRegressor(
    n_estimators=422, 
    max_depth=23, 
    min_samples_leaf=1,
    min_samples_split=2, 
    max_features=0.458,
    n_jobs=-1, 
    random_state=42
)
#{'n_estimators': 422, 'max_depth': 23, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.4576285822812347}

In [30]:
voting = VotingRegressor([
    #('hgb', hgb_model),
    ('et',  et_model),
    ('xgb', xgb_model),
    ('rf', rf_model)
])
voting.fit(X_train_pca, y_train)

# Evaluation
preds = voting.predict(X_test_pca)
r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n--- Leakage-Free Results ---")
print(f"R-squared: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


--- Leakage-Free Results ---
R-squared: 0.6770
RMSE: 11.1012


# K-Fold

In [31]:
from sklearn.model_selection import KFold
import numpy as np

n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

r2_scores   = []
rmse_scores = []

# Separate month columns before the loop
month_cols = ['Month_1','Month_2','Month_3','Month_4','Month_5','Month_6',
              'Month_7','Month_8','Month_9','Month_10','Month_11','Month_12']

X_month = X[month_cols].copy()
X_no_month = X.drop(columns=month_cols)

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    # Split numeric and month separately
    X_fold_train, X_fold_test       = X_no_month.iloc[train_idx], X_no_month.iloc[test_idx]
    month_fold_train, month_fold_test = X_month.iloc[train_idx].reset_index(drop=True), \
                                        X_month.iloc[test_idx].reset_index(drop=True)
    y_fold_train = Y.iloc[train_idx].reset_index(drop=True)
    y_fold_test  = Y.iloc[test_idx].reset_index(drop=True)

    # Impute → Scale → PCA (numeric only)
    imputer     = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_fold_train)
    X_test_imp  = imputer.transform(X_fold_test)

    scaler      = StandardScaler()
    X_train_ss  = scaler.fit_transform(X_train_imp)
    X_test_ss   = scaler.transform(X_test_imp)

    pca         = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_ss)
    X_test_pca  = pca.transform(X_test_ss)

    # Convert PCA output to DataFrame and attach month columns
    n_pcs = X_train_pca.shape[1]
    pc_cols = [f'PC{i+1}' for i in range(n_pcs)]

    X_train_final = pd.concat([
        pd.DataFrame(X_train_pca, columns=pc_cols),
        month_fold_train
    ], axis=1)
    X_test_final = pd.concat([
        pd.DataFrame(X_test_pca, columns=pc_cols),
        month_fold_test
    ], axis=1)

    # Train & evaluate
    voting = VotingRegressor([
        #('hgb', hgb_model),
        ('et',  et_model),
        ('xgb', xgb_model),
        ('rf', rf_model)
    ])
    voting.fit(X_train_final, y_fold_train)

    preds = voting.predict(X_test_final)
    r2_scores.append(r2_score(y_fold_test, preds))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_test, preds)))

    print(f"Fold {fold}  →  R²: {r2_scores[-1]:.4f}  |  RMSE: {rmse_scores[-1]:.4f}")

print(f"\n--- {n_splits}-Fold Summary ---")
print(f"R²   mean: {np.mean(r2_scores):.4f}  ±  {np.std(r2_scores):.4f}")
print(f"RMSE mean: {np.mean(rmse_scores):.4f}  ±  {np.std(rmse_scores):.4f}")

Fold 1  →  R²: 0.6624  |  RMSE: 11.1078
Fold 2  →  R²: 0.6548  |  RMSE: 11.6850
Fold 3  →  R²: 0.6724  |  RMSE: 10.8475
Fold 4  →  R²: 0.6637  |  RMSE: 11.2391

--- 4-Fold Summary ---
R²   mean: 0.6633  ±  0.0062
RMSE mean: 11.2199  ±  0.3033
